KPI-1 Encounter Mix by Encounter Class

In [0]:
%sql
create or replace view encounter_class_kpi as
with calendar as (
  select
    *
  from
    gold_medical_data.azure_blob_storage.dim_calendar
),
encounters as (
  select
    *
  from
    gold_medical_data.azure_blob_storage.dim_encounters
)
select
  year(e.encounter_start) as encounter_year,
  c.quater,
  e.encounter_class,
  count(*) as total_encounters,
  round(
    (total_encounters / (sum(count(*)) over (partition by year(e.encounter_start), c.quater))) * 100,2
  ) as total_encounters_percentage
from
  encounters e
    join calendar c
      on month(e.encounter_start) = c.month
group by
  encounter_year,
  c.quater,
  e.encounter_class
order by
  encounter_year,
  c.quater,
  e.encounter_class;

KPI-2 Encounters Over 24 Hours vs Under 24 Hours (%)

In [0]:
%sql
create or replace view encounters_durations_kpi as 
with encounter_durations AS (
  select
    date_format(encounter_start, 'yyyy-MM') as encounter_month,
    case
      when
        (unix_timestamp(encounter_stop) - unix_timestamp(encounter_start)) / 3600 > 24
      then
        'More than 24 hours'
      else '24 hours or less'
    end AS duration_category
  from
    gold_medical_data.azure_blob_storage.dim_encounters
  where
    encounter_start is not null
    and encounter_stop is not null
)
select
  encounter_month,
  duration_category,
  count(*) AS encounter_count,
  round((encounter_count / (sum(count(*)) over (partition by encounter_month)))*100,2) as monthly_percentage
from
  encounter_durations
group by
  encounter_month,
  duration_category
order by
  encounter_month asc,
  duration_category;

KPI-3 Zero Payer Coverage Rate  

In [0]:
%sql
create or replace view payer_coverage_kpi as
with fact_encounters as (
  select
    encounter_id,
    payer_coverage,
    case
      when payer_coverage > 0 then 1
      else 0
    end as has_payer_coverage
  from
    gold_medical_data.azure_blob_storage.fact_encounters
),
dim_encounters as (
  select
    encounter_id,
    date_format(encounter_stop, 'yyyy-MM') as encounter_month
  from
    gold_medical_data.azure_blob_storage.dim_encounters
)
select
  d.encounter_month,
  f.has_payer_coverage,
  count(*) as total_encounters,
  round(
    total_encounters / (sum(count(*)) over (partition by d.encounter_month)) * 100,
    2
  ) as total_encounters_percentage
from
  dim_encounters d
    join fact_encounters f
      on d.encounter_id = f.encounter_id
group by
  d.encounter_month,
  f.has_payer_coverage
order by
  d.encounter_month,
  f.has_payer_coverage;

KPI-4 Top 10 Procedures by Highest Average Base Cost

In [0]:
%sql
create or replace view avg_procedure_cost_kpi as
with calendar as (
  select * from
    gold_medical_data.azure_blob_storage.dim_calendar
),
procedures as (
  select dp.encounter_id,dp.procedure_code,c.half_yearly from
    gold_medical_data.azure_blob_storage.dim_procedures dp join 
    calendar c on month(dp.procedure_start) = c.month
)
select
  p.half_yearly,
  p.procedure_code,
  round(avg(fp.base_procedure_cost), 2) as avg_cost,
  count(*) as total_procedures
from
  procedures p
    join gold_medical_data.azure_blob_storage.fact_procedures fp
      on p.encounter_id = fp.encounter_id
group by
  p.half_yearly,
  p.procedure_code
order by
  avg_cost desc,
  total_procedures desc
limit 10;

KPI-5 Average total claim cost for encounters, broken down by payer

In [0]:
%sql
create or replace view avg_total_claim_cost_kpi as
select
  de.payer_id,
  de.payer_name,
  de.encounter_class,
  de.encounter_code,
  round(avg(fe.total_claim_cost),2) as avg_total_claim_cost
from
  gold_medical_data.azure_blob_storage.dim_encounters de join 
  gold_medical_data.azure_blob_storage.fact_encounters fe on de.encounter_id = fe.encounter_id
group by
  de.payer_id,
  de.payer_name,
  de.encounter_class,
  de.encounter_code
order by
  de.payer_id,
  de.payer_name,
  de.encounter_class,
  de.encounter_code;

KPI-6 30-Day Patient Readmission Rate

In [0]:
%sql
create or replace view readmission_rate_kpi as
with encounters as (
  select
    patient_id,
    encounter_start,
    encounter_stop,
    lag(encounter_start) over (
        partition by patient_id
        order by encounter_start, encounter_stop
      ) as previous_encounter_start,
    lag(encounter_stop) over (
        partition by patient_id
        order by encounter_start, encounter_stop
      ) as previous_encounter_stop
  from
    gold_medical_data.azure_blob_storage.dim_encounters
),
restructured_encounters as (
  select
    patient_id,
    date_format(encounter_stop, 'yyyy-MM') as encounter_month,
    case
      when encounter_start >= previous_encounter_stop then 'encounter_eligible'
      else 'encounter_not_eligible'
    end as encounter_eligible,
    datediff(encounter_start, previous_encounter_stop) as duration_between_encounters,
    case
      when duration_between_encounters <= 30 then 'readmission_eligible'
      else 'readmission_not_eligible'
    end as readmission_eligible
  from
    encounters
  where
    previous_encounter_stop is not null
  order by
    patient_id,
    encounter_month
),
readmission_encounters as (
  select
    patient_id,
    encounter_month,
    count(*) as readmission_count
  from
    restructured_encounters
  where
    readmission_eligible = 'readmission_eligible'
  group by
    patient_id,
    encounter_month
),
eligible_encounters as (
  select
    patient_id,
    encounter_month,
    count(*) as encounter_eligible_count
  from
    restructured_encounters
  where
    encounter_eligible = 'encounter_eligible'
  group by
    patient_id,
    encounter_month
)
select
  re.patient_id,
  re.encounter_month,
  re.readmission_count,
  ee.encounter_eligible_count,
  case
    when
      round((re.readmission_count / ee.encounter_eligible_count) * 100, 2) <= 100
    then
      round((re.readmission_count / ee.encounter_eligible_count) * 100, 2)
    else 0
  end as readmission_rate
from
  readmission_encounters re
    join eligible_encounters ee
      on re.patient_id = ee.patient_id
      and re.encounter_month = ee.encounter_month
order by
  re.patient_id,
  re.encounter_month;